# 03 — Results Visualization

Loads the JSON metrics + PNG plots produced by `scripts/run_eval.py` and renders the final comparison: DriftGuard vs. the rule-based baseline, plus the held-out generalisation test.

Run, in order:
```bash
python scripts/generate_benchmark.py
python scripts/train_detector.py
python scripts/run_eval.py
```

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RESULTS_DIR = Path("../results")
metrics_files = sorted(RESULTS_DIR.glob("*_metrics.json"))
print(f"Found {len(metrics_files)} result files: {[f.name for f in metrics_files]}")

In [ ]:
records = []
for f in metrics_files:
    with f.open() as fh:
        m = json.load(fh)
    records.append({
        "detector": m["detector_name"],
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "roc_auc_macro": m.get("roc_auc_macro"),
        "FAR": m["FAR"],
        "MAR": m["MAR"],
        "eval_time_sec": m["eval_time_sec"],
    })

summary_df = pd.DataFrame(records).sort_values("f1_macro", ascending=False)
summary_df

## Headline comparison

DriftGuard's three-way classification should substantially reduce the **False Alarm Rate** on benign drift compared to the rule-based baseline, while keeping the **Missed Attack Rate** on malicious writes low — the two competing failure modes this project targets.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(summary_df))
width = 0.35
ax.bar([i - width/2 for i in x], summary_df["FAR"], width, label="FAR (benign flagged as malicious)", color="#e63946")
ax.bar([i + width/2 for i in x], summary_df["MAR"], width, label="MAR (malicious missed)", color="#457b9d")
ax.set_xticks(list(x))
ax.set_xticklabels(summary_df["detector"], rotation=20, ha="right")
ax.set_ylabel("Rate")
ax.set_title("FAR vs. MAR by detector")
ax.legend()
plt.tight_layout()
plt.show()

## Saved plots from `run_eval.py`

In [ ]:
for name in [
    "driftguard_confusion_matrix.png",
    "driftguard_roc_curves.png",
    "driftguard_family_breakdown.png",
    "rule_based_confusion_matrix.png",
    "held_out_confusion_matrix.png",
]:
    path = RESULTS_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print(f"(missing: {name} — run scripts/run_eval.py)")

## Held-out generalisation check

The held-out result (`DriftGuard (held-out)`) is evaluated on attack families the classifier never trained on (`slow_drip`, `indirect_tool_injection`). A large F1/MAR gap between the in-distribution and held-out rows below indicates the detector is memorising attack templates rather than learning transferable signal.

In [ ]:
summary_df[summary_df["detector"].isin(["DriftGuard", "DriftGuard (held-out)"])]